In [ ]:
# ==============================================================================
# ETL 3 - Malha Cicloviária do Recife
# Bloco 1: Importações e Configurações
# ==============================================================================

# --- Importações Padrão ---
import pandas as pd
import numpy as np
import requests
import json
import os
import re
import unicodedata
import logging
from datetime import datetime
from io import BytesIO

# --- Importações Geoespaciais ---
import geopandas as gpd
from shapely.geometry import LineString, MultiLineString, Point
from shapely.ops import unary_union
import pyproj

# --- Google Drive ---
from google.colab import drive

# ==============================================================================
# CONFIGURAÇÕES GERAIS
# ==============================================================================

# 1. Montar o Google Drive
drive.mount('/content/drive')

# 2. Definir pasta de trabalho no Drive
DRIVE_BASE_PATH = "/content/drive/MyDrive/Backup_ETL3_Recife"
os.makedirs(DRIVE_BASE_PATH, exist_ok=True)

# 3. Criar subpastas organizadas
DIR_RAW = os.path.join(DRIVE_BASE_PATH, "raw")
DIR_PROCESSED = os.path.join(DRIVE_BASE_PATH, "processed")
DIR_REPORTS = os.path.join(DRIVE_BASE_PATH, "reports")

for directory in [DIR_RAW, DIR_PROCESSED, DIR_REPORTS]:
    os.makedirs(directory, exist_ok=True)

# 4. Configurar logging
logging.basicConfig(
    level=logging.INFO,
    format='%(asctime)s - %(levelname)s - %(message)s',
    handlers=[
        logging.FileHandler(os.path.join(DIR_REPORTS, 'etl3_log.txt')),
        logging.StreamHandler()
    ]
)

# ==============================================================================
# CONFIGURAÇÕES DAS FONTES DE DADOS
# ==============================================================================

# S3 - Malha Cicloviária do Recife
S3_URL = "https://dados.recife.pe.gov.br/api/3/action/package_show?id=malha-cicloviaria-do-recife"
S3_RESOURCE_ID = None  # Será descoberto via API

# S6 - Mapas de limites e divisões territoriais (bairros)
S6_URL = "https://dados.recife.pe.gov.br/api/3/action/package_show?id=mapas-de-limites-e-divisoes-territoriais"
S6_RESOURCE_ID = None  # Será descoberto via API

# ==============================================================================
# CONFIGURAÇÕES GEODESICAS
# ==============================================================================

# CRS de origem (geralmente WGS84 para GeoJSON)
CRS_ORIGEM = "EPSG:4326"

# CRS alvo (SIRGAS 2000 / UTM 25S - métrico para cálculos de área/distância)
CRS_ALVO = "EPSG:31985"

# ==============================================================================
# CONSTANTES
# ==============================================================================

TARGET_YEAR = 2024
TIMEOUT_REQUEST = 60  # segundos

# ==============================================================================
# MENSAGEM INICIAL
# ==============================================================================

print("="*60)
print("✅ BLOCO 1 CARREGADO COM SUCESSO!")
print("="*60)
print(f"📁 Pasta de trabalho: {DRIVE_BASE_PATH}")
print(f"📍 CRS Origem: {CRS_ORIGEM}")
print(f"📍 CRS Alvo: {CRS_ALVO}")
print(f"📅 Ano de referência: {TARGET_YEAR}")
print("="*60)

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
✅ BLOCO 1 CARREGADO COM SUCESSO!
📁 Pasta de trabalho: /content/drive/MyDrive/Backup_ETL3_Recife
📍 CRS Origem: EPSG:4326
📍 CRS Alvo: EPSG:31985
📅 Ano de referência: 2024


In [ ]:
# ==============================================================================
# BLOCO DE RESTAURAÇÃO DO ETL 3
# ==============================================================================

import os
import sqlite3
from google.colab import drive

# Montar o Google Drive
drive.mount('/content/drive')

# Configurar caminhos
DRIVE_BASE_PATH = "/content/drive/MyDrive/Backup_ETL3_Recife"
db_path = os.path.join(DRIVE_BASE_PATH, "ciclovias_recife.db")

# Verificar se o banco já existe no Drive
if os.path.exists(db_path):
    print("="*60)
    print("✅ BANCO DE DADOS ENCONTRADO NO DRIVE!")
    print("="*60)
    print(f"📁 Local: {db_path}")
    print("\n🔄 Carregando dados diretamente do banco SQLite...")

    # Conectar ao banco
    con = sqlite3.connect(db_path)

    # Carregar tabelas
    fato_segmentos = pd.read_sql("SELECT * FROM fato_segmentos_cicloviarios", con)
    dim_bairro = pd.read_sql("SELECT * FROM dim_bairro", con)
    dim_tipo = pd.read_sql("SELECT * FROM dim_tipo", con)
    dim_fonte = pd.read_sql("SELECT * FROM dim_fonte", con)

    con.close()

    print(f"✅ Dados restaurados com sucesso!")
    print(f"   - Fato segmentos: {len(fato_segmentos)} registros")
    print(f"   - Dimensão bairro: {len(dim_bairro)} registros")
    print(f"   - Dimensão tipo: {len(dim_tipo)} registros")
    print(f"   - Dimensão fonte: {len(dim_fonte)} registros")
    print("\n💡 Você pode pular os blocos de extração e processamento geoespacial.")
    print("   Prossega diretamente para as análises e indicadores.")
    print("="*60)

    # Flag para indicar que os dados foram restaurados
    DADOS_RESTAURADOS = True

else:
    print("="*60)
    print("⚠️ BANCO DE DADOS NÃO ENCONTRADO")
    print("="*60)
    print(f"📁 Local esperado: {db_path}")
    print("\n❗ Será necessário executar o pipeline completo:")
    print("   1. Download dos GeoJSONs (S3 e S6)")
    print("   2. Reprojeção e cálculo de comprimentos")
    print("   3. Interseção espacial com bairros")
    print("   4. Cálculo de indicadores e carga no banco")
    print("\n💡 Após a primeira execução, os dados serão salvos no Drive")
    print("   e você poderá restaurá-los nas próximas execuções.")
    print("="*60)

    # Flag para indicar que precisa executar o pipeline
    DADOS_RESTAURADOS = False

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
✅ BANCO DE DADOS ENCONTRADO NO DRIVE!
📁 Local: /content/drive/MyDrive/Backup_ETL3_Recife/ciclovias_recife.db

🔄 Carregando dados diretamente do banco SQLite...
✅ Dados restaurados com sucesso!
   - Fato segmentos: 367 registros
   - Dimensão bairro: 94 registros
   - Dimensão tipo: 5 registros
   - Dimensão fonte: 1 registros

💡 Você pode pular os blocos de extração e processamento geoespacial.
   Prossega diretamente para as análises e indicadores.


In [ ]:
# ==============================================================================
# ETL 3 - Malha Cicloviária do Recife
# Bloco 2: Descoberta dos Resource IDs via API CKAN
# ==============================================================================

import requests
import json

# ==============================================================================
# FUNÇÃO AUXILIAR: Consultar API CKAN
# ==============================================================================

def consultar_api_ckan(endpoint: str, params: dict = None) -> dict:
    """
    Consulta a API CKAN do Recife Dados com tratamento de erros.
    """
    url = f"https://dados.recife.pe.gov.br/api/3/action/{endpoint}"

    try:
        response = requests.get(url, params=params, timeout=TIMEOUT_REQUEST)
        response.raise_for_status()
        data = response.json()

        if not data.get("success"):
            raise ValueError(f"API retornou success=False: {data.get('error', 'Erro desconhecido')}")

        return data.get("result", {})

    except requests.exceptions.RequestException as e:
        logging.error(f"Erro na requisição à API: {e}")
        raise
    except json.JSONDecodeError as e:
        logging.error(f"Erro ao decodificar JSON da resposta: {e}")
        raise

# ==============================================================================
# CONSULTA S3: Malha Cicloviária do Recife
# ==============================================================================

print("="*60)
print("🔍 CONSULTANDO PACOTE S3: Malha Cicloviária do Recife")
print("="*60)

try:
    s3_metadata = consultar_api_ckan("package_show", {"id": "malha-cicloviaria-do-recife"})

    logging.info(f"✅ Pacote S3 encontrado: {s3_metadata.get('title')}")
    logging.info(f"   Descrição: {s3_metadata.get('notes', 'N/A')[:100]}...")

    # Listar resources disponíveis
    s3_resources = s3_metadata.get("resources", [])
    logging.info(f"   Total de resources: {len(s3_resources)}")

    # Identificar GeoJSON e CSV
    s3_geojson_resource = None
    s3_csv_resource = None

    for resource in s3_resources:
        fmt = resource.get("format", "").upper()
        name = resource.get("name", "").lower()

        if fmt == "GEOJSON" or "geojson" in name:
            s3_geojson_resource = resource
            logging.info(f"   📍 GeoJSON encontrado: {resource.get('name')}")
        elif fmt == "CSV" or "csv" in name:
            s3_csv_resource = resource
            logging.info(f"   📄 CSV encontrado: {resource.get('name')}")

    if s3_geojson_resource:
        S3_GEOJSON_URL = s3_geojson_resource.get("url")
        S3_GEOJSON_RESOURCE_ID = s3_geojson_resource.get("id")
        logging.info(f"   URL GeoJSON: {S3_GEOJSON_URL}")
    else:
        logging.warning("⚠️ Nenhum GeoJSON encontrado no pacote S3!")
        S3_GEOJSON_URL = None
        S3_GEOJSON_RESOURCE_ID = None

    if s3_csv_resource:
        S3_CSV_URL = s3_csv_resource.get("url")
        S3_CSV_RESOURCE_ID = s3_csv_resource.get("id")
        logging.info(f"   URL CSV: {S3_CSV_URL}")
    else:
        logging.warning("⚠️ Nenhum CSV encontrado no pacote S3!")
        S3_CSV_URL = None
        S3_CSV_RESOURCE_ID = None

except Exception as e:
    logging.error(f"❌ Erro ao consultar pacote S3: {e}")
    raise

# ==============================================================================
# CONSULTA S6: Mapas de Limites e Divisões Territoriais
# ==============================================================================

print("\n" + "="*60)
print("🔍 CONSULTANDO PACOTE S6: Mapas de Limites e Divisões Territoriais")
print("="*60)

try:
    s6_metadata = consultar_api_ckan("package_show", {"id": "mapas-de-limites-e-divisoes-territoriais"})

    logging.info(f"✅ Pacote S6 encontrado: {s6_metadata.get('title')}")
    logging.info(f"   Descrição: {s6_metadata.get('notes', 'N/A')[:100]}...")

    # Listar resources disponíveis
    s6_resources = s6_metadata.get("resources", [])
    logging.info(f"   Total de resources: {len(s6_resources)}")

    # Identificar GeoJSON dos bairros
    s6_bairros_resource = None

    for resource in s6_resources:
        fmt = resource.get("format", "").upper()
        name = resource.get("name", "").lower()

        # Procurar por GeoJSON que contenha "bairro" no nome
        if fmt == "GEOJSON" and "bairro" in name:
            s6_bairros_resource = resource
            logging.info(f"   📍 GeoJSON de Bairros encontrado: {resource.get('name')}")
            break

    # Se não encontrou com "bairro", pegar o primeiro GeoJSON
    if not s6_bairros_resource:
        for resource in s6_resources:
            fmt = resource.get("format", "").upper()
            if fmt == "GEOJSON":
                s6_bairros_resource = resource
                logging.info(f"   📍 GeoJSON genérico encontrado: {resource.get('name')}")
                break

    if s6_bairros_resource:
        S6_BAIRROS_URL = s6_bairros_resource.get("url")
        S6_BAIRROS_RESOURCE_ID = s6_bairros_resource.get("id")
        logging.info(f"   URL GeoJSON Bairros: {S6_BAIRROS_URL}")
    else:
        logging.warning("⚠️ Nenhum GeoJSON de bairros encontrado no pacote S6!")
        S6_BAIRROS_URL = None
        S6_BAIRROS_RESOURCE_ID = None

except Exception as e:
    logging.error(f"❌ Erro ao consultar pacote S6: {e}")
    raise

# ==============================================================================
# RESUMO DOS RECURSOS ENCONTRADOS
# ==============================================================================

print("\n" + "="*60)
print("📋 RESUMO DOS RECURSOS ENCONTRADOS")
print("="*60)

print(f"\nS3 - Malha Cicloviária:")
print(f"   GeoJSON: {'✅' if S3_GEOJSON_URL else '❌'} {S3_GEOJSON_URL or 'Não encontrado'}")
print(f"   CSV: {'✅' if S3_CSV_URL else '❌'} {S3_CSV_URL or 'Não encontrado'}")

print(f"\nS6 - Mapas de Limites:")
print(f"   Bairros GeoJSON: {'✅' if S6_BAIRROS_URL else '❌'} {S6_BAIRROS_URL or 'Não encontrado'}")

# Salvar metadados para referência futura
metadados_fontes = {
    "S3": {
        "titulo": s3_metadata.get("title"),
        "geojson_url": S3_GEOJSON_URL,
        "geojson_resource_id": S3_GEOJSON_RESOURCE_ID,
        "csv_url": S3_CSV_URL,
        "csv_resource_id": S3_CSV_RESOURCE_ID
    },
    "S6": {
        "titulo": s6_metadata.get("title"),
        "bairros_geojson_url": S6_BAIRROS_URL,
        "bairros_resource_id": S6_BAIRROS_RESOURCE_ID
    },
    "data_consulta": datetime.now().isoformat()
}

with open(os.path.join(DIR_REPORTS, "metadados_fontes.json"), "w", encoding="utf-8") as f:
    json.dump(metadados_fontes, f, ensure_ascii=False, indent=4)

print(f"\n💾 Metadados salvos em: {os.path.join(DIR_REPORTS, 'metadados_fontes.json')}")
print("="*60)

🔍 CONSULTANDO PACOTE S3: Malha Cicloviária do Recife

🔍 CONSULTANDO PACOTE S6: Mapas de Limites e Divisões Territoriais

📋 RESUMO DOS RECURSOS ENCONTRADOS

S3 - Malha Cicloviária:
   GeoJSON: ✅ https://dados.recife.pe.gov.br:443/dataset/8512f8d0-b0be-4880-9fbd-3d568c65ca1a/resource/c67fa150-8c69-4f6d-8fca-8204fdd61e35/download/rotas-ciclaveis-do-recife-conecta-.geojson
   CSV: ✅ https://dados.recife.pe.gov.br:443/dataset/8512f8d0-b0be-4880-9fbd-3d568c65ca1a/resource/6d2e322b-70ce-424c-93fa-073b02ff4b17/download/rotas-operacionais-domingos-e-feriados.csv

S6 - Mapas de Limites:
   Bairros GeoJSON: ✅ https://dados.recife.pe.gov.br:443/dataset/ce23d3f4-7474-44d3-b310-19f3afcebf4a/resource/d5f956ce-7e1f-4c74-839b-06cb490c3721/download/limites-dos-bairros-2023.geojson

💾 Metadados salvos em: /content/drive/MyDrive/Backup_ETL3_Recife/reports/metadados_fontes.json


In [ ]:
# ==============================================================================
# ETL 3 - Malha Cicloviária do Recife
# Bloco 3: Download e Validação dos GeoJSONs
# ==============================================================================

import geopandas as gpd
import requests

print("="*60)
print("📥 BAIXANDO E VALIDANDO ARQUIVOS GEOSPACIAIS")
print("="*60)

# ==============================================================================
# 1. Download e Carregamento do S3 (Malha Cicloviária)
# ==============================================================================
print("\n🚴 Processando S3: Malha Cicloviária...")

try:
    # Download do GeoJSON
    print("   Baixando GeoJSON da malha cicloviária...")
    response_s3 = requests.get(S3_GEOJSON_URL, timeout=TIMEOUT_REQUEST)
    response_s3.raise_for_status()

    # Salvar na pasta raw
    s3_raw_path = os.path.join(DIR_RAW, "malha_cicloviaria_raw.geojson")
    with open(s3_raw_path, "wb") as f:
        f.write(response_s3.content)
    print(f"   ✅ Salvo em: {s3_raw_path}")

    # Carregar como GeoDataFrame
    gdf_ciclovias = gpd.read_file(s3_raw_path)
    print(f"   ✅ Carregado: {len(gdf_ciclovias)} feições (segmentos de rota)")

    # Validar CRS
    print(f"   📍 CRS de origem: {gdf_ciclovias.crs}")
    if gdf_ciclovias.crs is None:
        logging.warning("⚠️ CRS não definido no GeoJSON. Assumindo EPSG:4326 (WGS84).")
        gdf_ciclovias.set_crs(epsg=4326, inplace=True)

    # Verificar geometrias inválidas ou nulas
    geometrias_nulas = gdf_ciclovias.geometry.isna().sum()
    geometrias_invalidas = (~gdf_ciclovias.geometry.is_valid).sum()

    if geometrias_nulas > 0:
        logging.warning(f"⚠️ {geometrias_nulas} geometrias nulas encontradas.")
    if geometrias_invalidas > 0:
        logging.warning(f"⚠️ {geometrias_invalidas} geometrias inválidas encontradas.")
        # Tentar corrigir geometrias inválidas (buffer 0 é um truque comum e seguro)
        gdf_ciclovias.geometry = gdf_ciclovias.geometry.buffer(0)
        print("   🛠️ Correção de geometrias inválidas aplicada (buffer 0).")

except Exception as e:
    logging.error(f"❌ Erro ao processar S3: {e}")
    raise

# ==============================================================================
# 2. Download e Carregamento do S6 (Limites dos Bairros)
# ==============================================================================
print("\n🗺️ Processando S6: Limites dos Bairros...")

try:
    # Download do GeoJSON
    print("   Baixando GeoJSON dos limites dos bairros...")
    response_s6 = requests.get(S6_BAIRROS_URL, timeout=TIMEOUT_REQUEST)
    response_s6.raise_for_status()

    # Salvar na pasta raw
    s6_raw_path = os.path.join(DIR_RAW, "limites_bairros_raw.geojson")
    with open(s6_raw_path, "wb") as f:
        f.write(response_s6.content)
    print(f"   ✅ Salvo em: {s6_raw_path}")

    # Carregar como GeoDataFrame
    gdf_bairros = gpd.read_file(s6_raw_path)
    print(f"   ✅ Carregado: {len(gdf_bairros)} feições (polígonos de bairros)")

    # Validar CRS
    print(f"   📍 CRS de origem: {gdf_bairros.crs}")
    if gdf_bairros.crs is None:
        logging.warning("⚠️ CRS não definido no GeoJSON. Assumindo EPSG:4326 (WGS84).")
        gdf_bairros.set_crs(epsg=4326, inplace=True)

    # Verificar geometrias inválidas ou nulas
    geometrias_nulas_b = gdf_bairros.geometry.isna().sum()
    geometrias_invalidas_b = (~gdf_bairros.geometry.is_valid).sum()

    if geometrias_nulas_b > 0:
        logging.warning(f"⚠️ {geometrias_nulas_b} geometrias nulas encontradas.")
    if geometrias_invalidas_b > 0:
        logging.warning(f"⚠️ {geometrias_invalidas_b} geometrias inválidas encontradas.")
        gdf_bairros.geometry = gdf_bairros.geometry.buffer(0)
        print("   🛠️ Correção de geometrias inválidas aplicada (buffer 0).")

except Exception as e:
    logging.error(f"❌ Erro ao processar S6: {e}")
    raise

# ==============================================================================
# 3. Resumo da Validação
# ==============================================================================
print("\n" + "="*60)
print("📋 RESUMO DA VALIDAÇÃO GEOMÉTRICA")
print("="*60)
print(f"S3 (Ciclovias): {len(gdf_ciclovias)} registros | CRS: {gdf_ciclovias.crs}")
print(f"S6 (Bairros):   {len(gdf_bairros)} registros | CRS: {gdf_bairros.crs}")

# Verificar se ambos estão no mesmo CRS para operações futuras
if gdf_ciclovias.crs == gdf_bairros.crs:
    print("\n✅ Ambos os GeoDataFrames estão no mesmo CRS. Pronto para operações espaciais.")
else:
    print("\n⚠️ CRSs diferentes! Será necessário reprojetar antes de realizar interseções.")

print("="*60)

📥 BAIXANDO E VALIDANDO ARQUIVOS GEOSPACIAIS

🚴 Processando S3: Malha Cicloviária...
   Baixando GeoJSON da malha cicloviária...
   ✅ Salvo em: /content/drive/MyDrive/Backup_ETL3_Recife/raw/malha_cicloviaria_raw.geojson
   ✅ Carregado: 79 feições (segmentos de rota)
   📍 CRS de origem: EPSG:4326

🗺️ Processando S6: Limites dos Bairros...
   Baixando GeoJSON dos limites dos bairros...
   ✅ Salvo em: /content/drive/MyDrive/Backup_ETL3_Recife/raw/limites_bairros_raw.geojson
   ✅ Carregado: 94 feições (polígonos de bairros)
   📍 CRS de origem: EPSG:4326

📋 RESUMO DA VALIDAÇÃO GEOMÉTRICA
S3 (Ciclovias): 79 registros | CRS: EPSG:4326
S6 (Bairros):   94 registros | CRS: EPSG:4326

✅ Ambos os GeoDataFrames estão no mesmo CRS. Pronto para operações espaciais.


In [ ]:
# ==============================================================================
# ETL 3 - Malha Cicloviária do Recife
# Bloco 4: Reprojeção, Cálculo de Comprimento e Inspeção de Tipos
# ==============================================================================

print("="*60)
print("🔄 REPROJEÇÃO E CÁLCULO DE COMPRIMENTO")
print("="*60)

# 1. Inspeção das colunas do GeoDataFrame de Ciclovias (S3)
print("\n🔍 Colunas disponíveis em gdf_ciclovias:")
print(gdf_ciclovias.columns.tolist())

print("\n🔍 Valores únicos nas colunas que podem indicar o tipo de rota:")
# Vamos verificar colunas comuns que costumam indicar o tipo (ajuste os nomes se necessário)
colunas_tipo = [col for col in gdf_ciclovias.columns if 'tipo' in col.lower() or 'class' in col.lower() or 'operac' in col.lower()]
for col in colunas_tipo:
    print(f"   - {col}: {gdf_ciclovias[col].dropna().unique().tolist()}")

# Se não encontrar colunas óbvias, vamos dar uma olhada nas primeiras linhas
print("\n📋 Amostra das 3 primeiras linhas:")
display(gdf_ciclovias.head(3))

# 2. Reprojeção para CRS Métrico (SIRGAS 2000 / UTM 25S - EPSG:31985)
print("\n📍 Reprojetando para EPSG:31985 (metros)...")
gdf_ciclovias_m = gdf_ciclovias.to_crs(epsg=31985)
gdf_bairros_m = gdf_bairros.to_crs(epsg=31985)

print(f"   ✅ gdf_ciclovias CRS: {gdf_ciclovias_m.crs}")
print(f"   ✅ gdf_bairros CRS: {gdf_bairros_m.crs}")

# 3. Cálculo do comprimento dos segmentos em metros e quilômetros
print("\n📏 Calculando comprimento dos segmentos...")
# O método .length em GeoSeries com CRS métrico retorna o comprimento em metros
gdf_ciclovias_m['comprimento_m'] = gdf_ciclovias_m.geometry.length
gdf_ciclovias_m['comprimento_km'] = gdf_ciclovias_m['comprimento_m'] / 1000

print(f"   ✅ Comprimento total da malha: {gdf_ciclovias_m['comprimento_km'].sum():.2f} km")
print(f"   ✅ Média por segmento: {gdf_ciclovias_m['comprimento_km'].mean():.2f} km")

# 4. Salvamento do arquivo processado (bruto mas com métricas)
processed_path = os.path.join(DIR_PROCESSED, "malha_cicloviaria_com_metricas.geojson")
gdf_ciclovias_m.to_file(processed_path, driver="GeoJSON")
print(f"\n💾 Arquivo com métricas salvo em: {processed_path}")

print("="*60)

🔄 REPROJEÇÃO E CÁLCULO DE COMPRIMENTO

🔍 Colunas disponíveis em gdf_ciclovias:
['Nome', 'Logradouro', 'Tipo', 'Sentido', 'Bairro', 'geometry']

🔍 Valores únicos nas colunas que podem indicar o tipo de rota:
   - Tipo: ['Ciclorrota,Ciclofaixa,Area Compartilhada', 'Ciclofaixa,Ciclorrota', 'Ciclofaixa', 'Ciclorrota,Ciclovia', 'Ciclovia', 'Ciclovia,Ciclofaixa', 'Ciclofaixa ,Ciclorrota,Ciclofaixa', 'Ciclofaixa,Ciclorrota,Area Compartilhada', 'Ciclorrota,Ciclofaixa,Ciclofaixa ', 'Ciclofaixa,Area Compartilhada', 'Ciclofaixa ', 'Ciclorrota', 'Area Compartilhada,Ciclorrota', 'Ciclorrota,Ciclofaixa', 'Ciclorrota,Area Compartilhada,Ciclofaixa', 'Ciclofaixa,Ciclorrota,Ciclovia', 'Ciclorrota,Ciclofaixa,Ciclovia,Area Compartilhada', 'Area Compartilhada,Ciclorrota,Ciclofaixa', 'Area Compartilhada,Ciclofaixa', 'Ciclorrota,Area Compartilhada,Area Compatilhada', 'Ciclofaixa,Ciclovia,Area Compartilhada', 'Ciclovia,Ciclofaixa,Area compartilhada', 'Ciclovia,Ciclofaixa,Area Compartilhada,Ciclorrota', 'Ciclo

,Nome,Logradouro,Tipo,Sentido,Bairro,geometry
0,Rota Afonso Olindense,"Av. Afonso Olindense,Praca Pinto Damaso,Rua Az...","Ciclorrota,Ciclofaixa,Area Compartilhada","Bidirecional,Unidirecional","Varzea,Cidade Universitaria","MULTILINESTRING ((-34.95958 -8.04862, -34.9592..."
1,Ciclofaixa Beberibe,"Av. Beberibe,Rua Alegre,Rua da Regeneracao,Rua...","Ciclofaixa,Ciclorrota","Unidirecional,Bidirecional","Agua Fria,Arruda,Bomba do Hemeterio","MULTILINESTRING ((-34.89726 -8.02403, -34.8974..."
2,Ciclofaixa Binario de Casa Amarela,"Estrada do Arraial,Estrada do Encanamento,Rua ...",Ciclofaixa,"Unidirecional,Bidirecional","Casa Amarela,Monteiro,Parnamirim,Tamarineira","MULTILINESTRING ((-34.90724 -8.03006, -34.9073..."



📍 Reprojetando para EPSG:31985 (metros)...
   ✅ gdf_ciclovias CRS: EPSG:31985
   ✅ gdf_bairros CRS: EPSG:31985

📏 Calculando comprimento dos segmentos...
   ✅ Comprimento total da malha: 186.09 km
   ✅ Média por segmento: 2.36 km

💾 Arquivo com métricas salvo em: /content/drive/MyDrive/Backup_ETL3_Recife/processed/malha_cicloviaria_com_metricas.geojson


In [ ]:
# ==============================================================================
# ETL 3 - Malha Cicloviária do Recife
# Bloco 5: Separação de Tipos e Interseção Espacial com Bairros (CORRIGIDO)
# ==============================================================================

print("="*60)
print("🔪 SEPARAÇÃO DE TIPOS E INTERSEÇÃO ESPACIAL COM BAIRROS")
print("="*60)

# 1. Limpar e explodir a coluna 'Tipo'
print("\n🔄 Explodindo a coluna 'Tipo' (separando tipos combinados)...")
# Remove espaços, converte para minúsculas e DIVIDE por vírgula para criar uma lista
gdf_ciclovias_m['Tipo_Limpo'] = gdf_ciclovias_m['Tipo'].str.lower().str.replace(r'\s+', '', regex=True).str.split(',')

# Agora o explode separará cada item da lista em uma nova linha
gdf_exploded = gdf_ciclovias_m.explode('Tipo_Limpo', ignore_index=True)
print(f"   ✅ Linhas após explosão: {len(gdf_exploded)}")

# Mapeamento para classificar como Permanente ou Operacional
def classificar_tipo(tipo):
    tipo = str(tipo).strip()
    if tipo in ['ciclovia', 'ciclofaixa']:
        return 'Permanente'
    elif tipo in ['ciclorrota', 'areacompartilhada', 'areacompatilhada']:
        return 'Operacional'
    else:
        return 'Não Classificado'

gdf_exploded['Categoria'] = gdf_exploded['Tipo_Limpo'].apply(classificar_tipo)

# Verificação rápida da classificação
print("   ✅ Distribuição inicial por Categoria:")
print(gdf_exploded['Categoria'].value_counts())

# 2. Preparar o GeoDataFrame de Bairros para a interseção
gdf_bairros_overlay = gdf_bairros_m[['EBAIRRNOMEOF', 'geometry']].copy()
gdf_bairros_overlay = gdf_bairros_overlay.rename(columns={'EBAIRRNOMEOF': 'Nome_Bairro_Oficial'})

# 3. Interseção Espacial com os Bairros
print("\n🗺️ Realizando interseção espacial com os limites dos bairros...")
gdf_intersecao = gpd.overlay(gdf_exploded, gdf_bairros_overlay, how='intersection')
print(f"   ✅ Linhas após interseção espacial: {len(gdf_intersecao)}")

# 4. Recalcular o comprimento dos segmentos fragmentados em metros e km
print("\n📏 Recalculando o comprimento dos segmentos recortados...")
gdf_intersecao['comprimento_m'] = gdf_intersecao.geometry.length
gdf_intersecao['comprimento_km'] = gdf_intersecao['comprimento_m'] / 1000

# 5. Salvar o resultado processado
processed_intersecao_path = os.path.join(DIR_PROCESSED, "malha_cicloviaria_por_bairro.geojson")
gdf_intersecao.to_file(processed_intersecao_path, driver="GeoJSON")
print(f"\n💾 Arquivo com interseção por bairro salvo em: {processed_intersecao_path}")

print("\n📊 Amostra dos dados processados:")
cols_to_show = ['Tipo_Limpo', 'Categoria', 'Nome_Bairro_Oficial', 'comprimento_km']
display(gdf_intersecao[cols_to_show].head(10))

print("="*60)
print("✅ BLOCO 5 CONCLUÍDO COM SUCESSO!")
print("="*60)

🔪 SEPARAÇÃO DE TIPOS E INTERSEÇÃO ESPACIAL COM BAIRROS

🔄 Explodindo a coluna 'Tipo' (separando tipos combinados)...
   ✅ Linhas após explosão: 154
   ✅ Distribuição inicial por Categoria:
Categoria
Permanente     88
Operacional    66
Name: count, dtype: int64

🗺️ Realizando interseção espacial com os limites dos bairros...
   ✅ Linhas após interseção espacial: 367

📏 Recalculando o comprimento dos segmentos recortados...

💾 Arquivo com interseção por bairro salvo em: /content/drive/MyDrive/Backup_ETL3_Recife/processed/malha_cicloviaria_por_bairro.geojson

📊 Amostra dos dados processados:


,Tipo_Limpo,Categoria,Nome_Bairro_Oficial,comprimento_km
0,ciclorrota,Operacional,Curado,0.980448
1,ciclorrota,Operacional,Várzea,3.389050
2,ciclorrota,Operacional,Cidade Universitária,1.423288
3,ciclofaixa,Permanente,Curado,0.980448
4,ciclofaixa,Permanente,Várzea,3.389050
5,ciclofaixa,Permanente,Cidade Universitária,1.423288
6,areacompartilhada,Operacional,Curado,0.980448
7,areacompartilhada,Operacional,Várzea,3.389050
8,areacompartilhada,Operacional,Cidade Universitária,1.423288
9,ciclofaixa,Permanente,Arruda,0.413007


✅ BLOCO 5 CONCLUÍDO COM SUCESSO!


In [ ]:
# ==============================================================================
# ETL 3 - Malha Cicloviária do Recife
# Bloco 6: Cálculo dos Indicadores (km por tipo/bairro e densidade da malha)
# ==============================================================================

print("="*60)
print("📊 CÁLCULO DOS INDICADORES DE MALHA CICLOVIÁRIA")
print("="*60)

# 1. Calcular a área de cada bairro em km² (usando o GeoDataFrame de bairros em metros)
print("\n📏 Calculando a área dos bairros...")
gdf_bairros_m['area_m2'] = gdf_bairros_m.geometry.area
gdf_bairros_m['area_km2'] = gdf_bairros_m['area_m2'] / 1_000_000

# Criar um dataframe de resumo dos bairros com suas áreas
df_areas_bairros = gdf_bairros_m[['EBAIRRNOMEOF', 'area_km2']].copy()
df_areas_bairros = df_areas_bairros.rename(columns={'EBAIRRNOMEOF': 'Nome_Bairro_Oficial'})

# 2. Agregar os comprimentos das ciclovias por Bairro e Tipo
print("🔄 Agregando quilômetros por bairro e tipo de rota...")

indicador_km_bairro_tipo = (
    gdf_intersecao
    .groupby(['Nome_Bairro_Oficial', 'Categoria', 'Tipo_Limpo'], dropna=False)['comprimento_km']
    .sum()
    .reset_index()
    .sort_values(['Nome_Bairro_Oficial', 'comprimento_km'], ascending=[True, False])
)

# 3. Calcular o total de km por bairro (para facilitar o cálculo da densidade)
indicador_km_bairro_total = (
    gdf_intersecao
    .groupby('Nome_Bairro_Oficial', dropna=False)['comprimento_km']
    .sum()
    .reset_index()
    .rename(columns={'comprimento_km': 'km_total_bairro'})
)

# 4. Unir os dados de comprimento com a área dos bairros para calcular a densidade
print("📐 Calculando a densidade da malha (km/km²)...")

# Merge com a área
indicador_densidade = indicador_km_bairro_total.merge(df_areas_bairros, on='Nome_Bairro_Oficial', how='left')

# Calcular densidade (tratando possíveis divisões por zero ou áreas nulas)
indicador_densidade['area_km2'] = indicador_densidade['area_km2'].fillna(0)
indicador_densidade['densidade_km_por_km2'] = (
    indicador_densidade['km_total_bairro'] / indicador_densidade['area_km2']
).round(4)

# Ordenar por densidade decrescente
indicador_densidade = indicador_densidade.sort_values('densidade_km_por_km2', ascending=False)

# 5. Resumo geral por Tipo de Categoria (Permanente vs Operacional)
print("📋 Agregando por categoria geral (Permanente vs Operacional)...")
indicador_categoria_geral = (
    gdf_intersecao
    .groupby('Categoria', dropna=False)['comprimento_km']
    .sum()
    .reset_index()
    .rename(columns={'comprimento_km': 'km_total'})
    .sort_values('km_total', ascending=False)
)

indicador_categoria_geral['percentual'] = (
    indicador_categoria_geral['km_total'] / indicador_categoria_geral['km_total'].sum() * 100
).round(2)

# 6. Salvar os indicadores processados
print("\n💾 Salvando indicadores...")
indicador_km_bairro_tipo.to_csv(os.path.join(DIR_PROCESSED, "indicador_km_bairro_tipo.csv"), index=False, encoding="utf-8-sig")
indicador_densidade.to_csv(os.path.join(DIR_PROCESSED, "indicador_densidade_bairro.csv"), index=False, encoding="utf-8-sig")
indicador_categoria_geral.to_csv(os.path.join(DIR_PROCESSED, "indicador_categoria_geral.csv"), index=False, encoding="utf-8-sig")

print("\n📊 Amostra dos Indicadores de Densidade (Top 10 bairros):")
display(indicador_densidade.head(10))

print("\n📊 Resumo por Categoria (Permanente vs Operacional):")
display(indicador_categoria_geral)

print("="*60)
print("✅ BLOCO 6 CONCLUÍDO COM SUCESSO!")
print("="*60)

📊 CÁLCULO DOS INDICADORES DE MALHA CICLOVIÁRIA

📏 Calculando a área dos bairros...
🔄 Agregando quilômetros por bairro e tipo de rota...
📐 Calculando a densidade da malha (km/km²)...
📋 Agregando por categoria geral (Permanente vs Operacional)...

💾 Salvando indicadores...

📊 Amostra dos Indicadores de Densidade (Top 10 bairros):


,Nome_Bairro_Oficial,km_total_bairro,area_km2,densidade_km_por_km2
60,Torreão,5.639457,0.195972,28.7769
28,Ilha do Leite,3.338065,0.257456,12.9656
56,Soledade,3.891054,0.322416,12.0684
54,Santo Amaro,42.746177,3.746344,11.4101
48,Ponto de Parada,1.877209,0.185226,10.1347
22,Espinheiro,7.515414,0.760526,9.8819
46,Peixinhos,3.309748,0.339010,9.7630
2,Arruda,9.598563,1.003687,9.5633
55,Santo Antônio,7.992726,0.838234,9.5352
27,Ilha Joana Bezerra,10.955961,1.163966,9.4126



📊 Resumo por Categoria (Permanente vs Operacional):


,Categoria,km_total,percentual
1,Permanente,215.666482,53.93
0,Operacional,184.222188,46.07


✅ BLOCO 6 CONCLUÍDO COM SUCESSO!


In [ ]:
# ==============================================================================
# ETL 3 - Malha Cicloviária do Recife
# Bloco 7: Carga no Banco SQLite com Modelo Dimensional
# ==============================================================================

import sqlite3
import os

print("="*60)
print("🗄️ CARGA NO BANCO SQLITE COM MODELO DIMENSIONAL")
print("="*60)

# Caminho do banco no Drive
db_path = os.path.join(DRIVE_BASE_PATH, "ciclovias_recife.db")

# Conectar ao banco (cria se não existir)
con = sqlite3.connect(db_path)

# ==============================================================================
# DIMENSÃO BAIRRO
# ==============================================================================
print("\n📍 Criando dimensão bairro...")
dim_bairro = df_areas_bairros.copy().reset_index(drop=True)
dim_bairro.insert(0, "id_bairro", range(1, len(dim_bairro) + 1))
dim_bairro.to_sql("dim_bairro", con, if_exists="replace", index=False)
print(f"   ✅ {len(dim_bairro)} bairros carregados")

# ==============================================================================
# DIMENSÃO TIPO DE ROTA
# ==============================================================================
print("\n🚲 Criando dimensão tipo de rota...")
dim_tipo = gdf_intersecao[['Tipo_Limpo', 'Categoria']].drop_duplicates().reset_index(drop=True)
dim_tipo.insert(0, "id_tipo", range(1, len(dim_tipo) + 1))
dim_tipo.to_sql("dim_tipo", con, if_exists="replace", index=False)
print(f"   ✅ {len(dim_tipo)} tipos de rota carregados")

# ==============================================================================
# DIMENSÃO FONTE
# ==============================================================================
print("\n📋 Criando dimensão fonte...")
dim_fonte = pd.DataFrame([{
    "id_fonte": 1,
    "fonte": "Portal de Dados Abertos do Recife",
    "conjunto": "Malha Cicloviária do Recife",
    "data_extracao": datetime.now().isoformat(),
    "formato": "GeoJSON"
}])
dim_fonte.to_sql("dim_fonte", con, if_exists="replace", index=False)
print(f"   ✅ 1 registro de fonte carregado")

# ==============================================================================
# FATO: SEGMENTOS CICLOVIÁRIOS
# ==============================================================================
print("\n📊 Criando fato_segmentos_cicloviarios...")

# Preparar o dataframe para merge
fato = gdf_intersecao.copy()

# Converter geometria para WKT (Well-Known Text) para armazenamento no SQLite
fato['geometry_wkt'] = fato.geometry.apply(lambda geom: geom.wkt if geom else None)

# Merge com dim_bairro
fato = fato.merge(dim_bairro, left_on='Nome_Bairro_Oficial', right_on='Nome_Bairro_Oficial', how='left')

# Merge com dim_tipo
fato = fato.merge(dim_tipo, on=['Tipo_Limpo', 'Categoria'], how='left')

# Adicionar id_fonte
fato['id_fonte'] = 1

# Selecionar colunas da fato
cols_fato = [
    "id_bairro", "id_tipo", "id_fonte",
    "Tipo_Limpo", "Categoria", "Nome_Bairro_Oficial",
    "comprimento_km", "geometry_wkt"
]

fato_segmentos = fato[cols_fato].copy()
fato_segmentos.insert(0, "id_segmento", range(1, len(fato_segmentos) + 1))

# Salvar no banco
fato_segmentos.to_sql("fato_segmentos_cicloviarios", con, if_exists="replace", index=False)
print(f"   ✅ {len(fato_segmentos)} segmentos carregados com geometria WKT")

# Fechar conexão
con.close()

print("\n" + "="*60)
print("✅ BANCO SQLITE CRIADO COM SUCESSO!")
print(f"📁 Local: {db_path}")
print("="*60)

🗄️ CARGA NO BANCO SQLITE COM MODELO DIMENSIONAL

📍 Criando dimensão bairro...
   ✅ 94 bairros carregados

🚲 Criando dimensão tipo de rota...
   ✅ 5 tipos de rota carregados

📋 Criando dimensão fonte...
   ✅ 1 registro de fonte carregado

📊 Criando fato_segmentos_cicloviarios...
   ✅ 367 segmentos carregados com geometria WKT

✅ BANCO SQLITE CRIADO COM SUCESSO!
📁 Local: /content/drive/MyDrive/Backup_ETL3_Recife/ciclovias_recife.db
